<br><br>

## 🟢 자연어 3  


### 🟡 다운로드  

In [1]:
import requests
import hashlib
import subprocess
import re
import string
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization
import os, pathlib, shutil, random

# 공식 Stanford URL 및 파일명
url = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
file_name = "aclImdb_v1.tar.gz"

# SHA-256 해시값을 미리 알고 있다면 여기에 입력 (선택)
# 공식적으로 제공되지 않아서, 처음 1번만 출력 후 복사해두면 좋음
expected_sha256 = None  # 예: "c40f8aef5bd12b2e68e7bd7f395e5b889bb0f8f1..."


# SHA-256 해시 계산 함수
def get_sha256(file_path):
    sha256 = hashlib.sha256()
    with open(file_path, "rb") as f:
        for chunk in iter(lambda: f.read(4096), b""):
            sha256.update(chunk)
    return sha256.hexdigest()


# 다운로드 함수
def download():
    print("🔽 다운로드 시작:", file_name)
    response = requests.get(url, stream=True)
    with open(file_name, "wb") as file:
        for chunk in response.iter_content(chunk_size=8192):
            file.write(chunk)
    print("✅ 다운로드 완료!")

    # 해시 체크
    print("🔍 SHA-256 해시 계산 중...")
    actual_hash = get_sha256(file_name)
    print("📄 계산된 SHA-256 해시:", actual_hash)

    if expected_sha256:
        if actual_hash == expected_sha256:
            print("✅ 해시 일치: 파일이 안전합니다.")
        else:
            print("❌ 해시 불일치: 파일이 변조되었을 수 있습니다!")
            os.remove(file_name)
            raise ValueError("파일 무결성 검사 실패")


# 실행
download() # 다운로드

2025-08-01 00:22:33.869910: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-08-01 00:22:35.262444: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-08-01 00:22:35.905129: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754007756.478874    2298 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1754007756.618912    2298 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1754007757.993914    2298 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

🔽 다운로드 시작: aclImdb_v1.tar.gz
✅ 다운로드 완료!
🔍 SHA-256 해시 계산 중...
📄 계산된 SHA-256 해시: c40f74a18d3b61f90feba1e17730e0d38e8b97c05fde7008942e91923d1658fe


Bad pipe message: %s [b'\x92&)h{uN\xb3\xc6(nmp~J\xce\xee\xaf \xa9\x04Se\xa8\xe4\xfb\xe9\x1f\xae\xe3\xb6\xac\x06\xae\xa1\xbe\xacYI\\s\xfb\x0cP\x1b\x94F\x9a\xd8\xa9\x95\x00\x1a\xc0+\xc0/\xc0,\xc00\xcc\xa9\xcc\xa8\xc0\t\xc0\x13\xc0\n\xc0\x14\x13\x01\x13\x02\x13\x03\x01\x00\x05A\x00\x0b\x00\x02\x01\x00\xff\x01\x00\x01\x00\x00\x17\x00\x00\x00\x12\x00\x00\x00\x05\x00\x05\x01\x00\x00\x00\x00\x00\n\x00\x0c\x00\n\x11\xec\x00\x1d\x00\x17\x00\x18\x00\x19\x00\r\x00\x1a\x00\x18\x08\x04\x04\x03\x08\x07\x08\x05\x08\x06\x04\x01\x05\x01\x06\x01\x05\x03\x06\x03\x02\x01\x02\x03\x00+\x00\x05\x04\x03\x04\x03\x03\x003\x04\xea\x04\xe8\x11\xec\x04\xc0\x01\x90%\x82S\x8b\x18\xf0\x92$\xaa`\x92q\x8f\x97\xe4\xc3\xeduzf$a\x9c2</4\x90\x00\xecr\xff\x872\xde\x88\xc9\xb8\x08\xab\xc3\x00\x9e\x17\xe2Q\xc1\x9a\x7fR\\\xa8d\x8b{\x03q\x92\xbf\xb8:\xc6\xa2)\xaf\xe6%\x90\xeb;0\x84\xb2', b"\xaegh()t&r\x90M\x90\xfa\x1b\xac\xa5\xa43\xc0j\x85\xc3\xb2J\xd7\x1c\\\x9bB\x04\x91lc\xbbA\x95\x11\x0b\xcb4g\x1a*\x8dM%\x9d\xe6\xf1_\x1bS\x17

### 🟡 압축 해제  

In [ ]:
import tarfile
import os

file_name = "aclImdb_v1.tar.gz"  # 압축된 파일명
extract_to = "aclImdb"  # 압축을 풀 폴더 이름 (없으면 자동 생성됨)


# 압축 해제 함수
def extract_file(file_path, extract_to):
    if not os.path.exists(file_path):
        print(f"❌ 파일이 존재하지 않습니다: {file_path}")
        return

    print(f"📦 압축 해제 중: {file_path}")
    with tarfile.open(file_path, "r:gz") as tar:
        tar.extractall(path=extract_to)
    print(f"✅ 압축 해제 완료! → 폴더: {extract_to}/")


# 실행
extract_file(file_name, extract_to)

### 🟡 라벨링  

In [ ]:
# train => train과 validation으로 나눠야 한다. , train 폴더에 있는 unsup 폴더는 직접 지워냐 한다.
# 라벨이 2개 여야 한다.


# 라벨링
def labeling():
    base_dir = pathlib.Path("aclImdb")
    val_dir = base_dir / "val"  # pathlib 객체에  / "디렉토리" => 결과가 문자열이 아니다
    train_dir = base_dir / "train"

    for category in ("neg", "pos"):
        os.makedirs(val_dir / category)  # 디렉토리를 만들고
        files = os.listdir(
            train_dir / category
        )  # 해당 카테고리의 파일 목록을 모두 가져온다
        random.Random(1337).shuffle(
            files
        )  # 파일을 랜덤하게 섞어서 복사하려고 파일 목록을 모두 섞는다
        num_val_samples = int(0.2 * len(files))
        val_files = files[-num_val_samples:]  # 20%만 val폴더로 이동한다
        for fname in val_files:
            shutil.move(train_dir / category / fname, val_dir / category / fname)


labeling()

### 🟡  

In [ ]:
# 데이터셋을 활용해서 디렉토리로부터 파일을 불러와서 벡터화를 진행한다
import keras

batch_size = 32  # 한번에 읽어올 양
train_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/train", batch_size=batch_size  # 디렉토리명
)

val_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/val", batch_size=batch_size  # 디렉토리명
)

test_ds = keras.utils.text_dataset_from_directory(
    "aclImdb/test", batch_size=batch_size  # 디렉토리명
)
# 데이터셋은 알아서  inputs, targets 을 반복해서 갖고 온다. 우리한테 필요한거는 inputs만이다
for inputs, targets in train_ds:  # 실제 읽어오는 데이터 확인
    print("inputs.shape", inputs.shape)
    print("inputs.dtype", inputs.dtype)
    print("targets.shape", targets.shape)
    print("targets.dtype", targets.dtype)
    print("inputs[0]", inputs[:3])
    print("targets[0]", targets[:3])
    break  # 하나만 출력해보자
# 0이 부정 1이 긍정 -> 폴더명을 정렬해서 0,1,2 이런식으로 라벨링을 한다 neg -0 pos-1

In [ ]:
# 데이터셋이  문장과 라벨로 연결되어, 라벨은 이미 정수화되어서 오니까 이거 버리고, 문장만 벡터화를 해야 한다
text_vectorization = TextVectorization(
    max_tokens=20000,  # 자주 사용하는 단어 20000개만 지정함,
    output_mode="multi_hot",  # 벡터로, 각 리뷰마다 20000개의 요소를 갖는 배열이 만들어진다.
    # 배열에서 문장중에 단어가 있는곳은 1 아니면 0으로 바꿔온다
    # 멀티핫 인코딩 또는 BoW(Bag Of Word, 단어가방) 방식
)

# 가져온 train_ds 에서 문장만 필요하다
text_only_train_ds = train_ds.map(lambda x, y: x)  # 데이터셋으로부터 문장만 추출
text_vectorization.adapt(text_only_train_ds)  # 어휘사전 만들기

####  각 데이터별로 이 작업을 진행해야 한다.
# 유니그램 - 단어를 1개씩 가져오는 방식

# 멀티프로세싱 , 한번에 cpu코더 4개를 사용해서 작업을 수행한다.
binary_1gram_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=4
)
binary_1gram_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=4
)
binary_1gram_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=4
)

print("벡터화 후 ----------------------------------")
for inputs, targets in binary_1gram_train_ds:  # 실제 읽어오는 데이터 확인
    print("inputs.shape", inputs.shape)
    print("inputs.dtype", inputs.dtype)
    print("targets.shape", targets.shape)
    print("targets.dtype", targets.dtype)
    print("inputs[0]", inputs[:3])
    print("targets[0]", targets[:3])
    break  # 하나만 출력해보자

In [ ]:
# 모델 만들어서 반환하는 함수
from keras import layers, models


def getModel(max_tokens=20000, hidden_dim=16):
    inputs = keras.Input(shape=(max_tokens,))  # 입력층 만들기
    x = layers.Dense(hidden_dim, activation="relu")(inputs)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer="rmsprop", loss="binary_crossentropy", metrics=["accuracy"])

    return model


model = getModel()
model.summary()
callbacks = [keras.callbacks.ModelCheckpoint("binary_1gram.keras", save_best_only=True)]
# 적절한 시점에서 파일 저장하기

# cache : 데이터셋을 메모리에 캐싱한다. 첫번재 에포크에서 전처리를 한번만 하고 더이상 하지 않고 재사용을 한다
# 메모리에 들어갈 만큼 작은 데이터셋일때만 가능하다
model.fit(
    binary_1gram_train_ds.cache(),
    validation_data=binary_1gram_val_ds,
    epochs=10,
    callbacks=callbacks,
)

model = models.load_model("binary_1gram.keras")  # 학습한 내용 읽어보기
print("테스트셋 정확도 ", model.evaluate(binary_1gram_test_ds))